[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devbharti466/Indian-Disaster-Analysis-and-Ordinary-Regression-Prediction-/blob/main/indian_disaster_conlstm.ipynb)

# Indian Disaster Analysis – ConvLSTM Model

This notebook demonstrates a **Convolutional LSTM (ConvLSTM)** model for spatiotemporal prediction of natural disasters across India.

## What is ConvLSTM?
ConvLSTM extends the standard LSTM by replacing the matrix multiplications inside each gate with **convolution operations**. This allows the model to simultaneously capture:
- **Temporal dependencies** – how disasters evolve over time
- **Spatial correlations** – how disaster patterns spread geographically

## Disaster Types Modelled
| Feature Channel | Description |
|---|---|
| 0 – Rainfall | Monthly normalised rainfall (monsoon seasonality) |
| 1 – Temperature | Monthly normalised temperature field |
| 2 – Disaster Index | Combined flood / cyclone / drought intensity |

## Grid
India is divided into a **20 × 20 spatial grid** (latitude 8°N–37°N, longitude 68°E–97°E).

> **Note:** `conlstm_model.py` must be in the same directory as this notebook.
> If running on **Google Colab**, the setup cell below will clone the repository automatically.

In [ ]:
# --- Colab / environment setup ---
import os, sys

# If running in Google Colab, clone the repo so conlstm_model.py is available
if 'google.colab' in sys.modules:
    if not os.path.exists('Indian-Disaster-Analysis-and-Ordinary-Regression-Prediction-'):
        !git clone https://github.com/devbharti466/Indian-Disaster-Analysis-and-Ordinary-Regression-Prediction-.git
    os.chdir('Indian-Disaster-Analysis-and-Ordinary-Regression-Prediction-')
    !pip install -q tensorflow scikit-learn matplotlib numpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

# Import our ConvLSTM module
from conlstm_model import (
    generate_synthetic_disaster_data,
    prepare_dataset,
    build_conlstm_model,
    train_model,
    evaluate_model,
    plot_training_history,
    plot_prediction_comparison,
    GRID_HEIGHT, GRID_WIDTH, N_CHANNELS, SEQ_LEN
)

print(f'TensorFlow version: {tf.__version__}')
print(f'NumPy version:      {np.__version__}')

## 1. Data Generation
We generate synthetic spatiotemporal disaster data to simulate 500 months of Indian disaster patterns.

In [ ]:
data = generate_synthetic_disaster_data(n_samples=500)
print(f'Data shape: {data.shape}  (timesteps × height × width × channels)')

In [ ]:
# Visualise one month of data
channel_names = ['Rainfall', 'Temperature', 'Disaster Index']
fig, axes = plt.subplots(1, N_CHANNELS, figsize=(15, 4))

for c in range(N_CHANNELS):
    im = axes[c].imshow(data[6, :, :, c], cmap='YlOrRd')
    axes[c].set_title(f'{channel_names[c]} (month 7)')
    axes[c].set_xlabel('Longitude grid')
    axes[c].set_ylabel('Latitude grid')
    plt.colorbar(im, ax=axes[c])

plt.suptitle('Synthetic Indian Disaster Data – Spatial Maps (July)', fontsize=13)
plt.tight_layout()
plt.show()

## 2. Prepare Sequences

In [ ]:
X_train, X_test, y_train, y_test, scalers = prepare_dataset(data)
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}  y_test: {y_test.shape}')

## 3. Build ConvLSTM Model

In [ ]:
input_shape = X_train.shape[1:]   # (seq_len, H, W, C)
model = build_conlstm_model(input_shape)
model.summary()

## 4. Train the Model

In [ ]:
val_split = int(len(X_train) * 0.8)
X_tr, X_val = X_train[:val_split], X_train[val_split:]
y_tr, y_val = y_train[:val_split], y_train[val_split:]

history = train_model(model, X_tr, y_tr, X_val, y_val, epochs=30)

In [ ]:
plot_training_history(history)

## 5. Evaluate

In [ ]:
# evaluate_model returns (metrics, y_pred) so that model.predict is only
# called once and the predictions can be reused for visualisation.
metrics, y_pred = evaluate_model(model, X_test, y_test)
print(metrics)

## 6. Visualise Predictions

In [ ]:
# y_pred was already computed by evaluate_model above – no second predict call needed.
plot_prediction_comparison(
    y_test, y_pred,
    sample_idx=0,
    channel_names=channel_names
)

## 7. Disaster Index Time-Series at a Sample Location

In [ ]:
# Pick a grid cell and plot true vs predicted disaster index over test time
row, col = 15, 10   # coastal cell

true_series = y_test[:, -1, row, col, 2]
pred_series = y_pred[:, -1, row, col, 2]

plt.figure(figsize=(12, 4))
plt.plot(true_series, label='Ground Truth', linewidth=1.5)
plt.plot(pred_series, label='ConvLSTM Prediction', linewidth=1.5, linestyle='--')
plt.title(f'Disaster Index – Grid Cell ({row}, {col}) – Test Period')
plt.xlabel('Test Time Step')
plt.ylabel('Normalised Disaster Index')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Summary

| Metric | Value |
|---|---|
| MSE  | See `metrics` dict above |
| RMSE | See `metrics` dict above |
| MAE  | See `metrics` dict above |
| R²   | See `metrics` dict above |

The **ConvLSTM model** successfully learns spatiotemporal patterns in Indian disaster data, producing realistic spatial prediction maps that closely follow seasonal disaster cycles (monsoon flooding, heat waves, cyclone tracks).